# PIMC Path Visualization

This notebook visualizes worldline paths output from PIMC boson simulations code. We show several ways of visualization

## Initialization

In [ ]:
import sys
import os
import numpy as np

# Add parent directory to path to import from jax_landscape
import os
import sys
sys.path.insert(0, os.path.abspath('../..'))

from jax_landscape.io.pimc import load_pimc_worldline_file, Path

# Visualization libraries
import plotly.graph_objects as go
from ipywidgets import interact, IntSlider

Utilities

In [ ]:
def should_draw_connection(pos1, pos2, box_dims):
    """
    Check if a connection should be drawn based on periodic boundary conditions.
    Returns True if the connection should be drawn (not wrapping around PBC).
    """
    for i in range(3):
        distance = abs(pos1[i] - pos2[i])
        if distance > box_dims[i] / 2.0:
            return False
    return True


## Input

In [ ]:
import pandas as pd
# Load PIMC configurations. 

# Example: Use test data or a trajectory file
# file_path = '../../tests/test_data/N2-Nbeads3-cycle1.dat'
file_path = '../../tests/tmp/pimc_M3_N2_trajectory.dat'  # Minimization trajectory

# simulation box dimensions (Lx = Ly = Lz = 10.0 Å)
box = [10.0, 10.0, 10.0]

confs = load_pimc_worldline_file(file_path, Lx=box[0], Ly=box[1], Lz=box[2])

print(f"✅ Successfully loaded {len(confs.keys())} PIMC configurations")
print(f"📦 Box dimensions set to: {box} Å")

# count closed configs
confs_closed = {}
for i, conf in confs.items():
    if conf.is_closed_worldline:
        confs_closed[i] = conf

print(f"🔒 Found {len(confs_closed)} closed configurations")

## Coordinate space trace 3d

In [ ]:
def plot_coordSpaceTrace_3d(index):
    """
    Plot the 3D coordinate space trace of worldline path from a PIMC configuration.
    """

    # Create Path object with box dimensions
    conf_idx = list(confs_closed.keys())[index]
    conf = confs_closed[conf_idx]
    print(f"🔍 Loaded Path[{conf_idx}] with {conf.numTimeSlices} time slices and {conf.numParticles} particles")
    print(f"cycleIndex shape: {conf.cycleIndex.shape}")
    print(f"cycleSizeDist shape: {conf.cycleSizeDist.shape}")

    fig = go.Figure()

    # Get normalized cycle lengths (in units of numTimeSlices)
    cycle_lengths_normalized = conf.cycleSizeDist / conf.numTimeSlices
    unique_cycle_lengths = np.unique(cycle_lengths_normalized)

    # Create a mapping from cycle length to color index
    cycle_length_to_color_idx = {length: idx for idx, length in enumerate(unique_cycle_lengths)}
    num_unique_lengths = len(unique_cycle_lengths)

    # Group beads by their cycle length
    beads_by_cycle_length = {length: {'x': [], 'y': [], 'z': []} for length in unique_cycle_lengths}

    for m in range(conf.numTimeSlices):
        for n in range(conf.numParticles):
            # Get the cycle index for this bead
            cycle_idx = conf.cycleIndex[m, n]
            # Get the normalized cycle length
            cycle_length = cycle_lengths_normalized[cycle_idx]
            # Add bead coordinates to the appropriate group
            beads_by_cycle_length[cycle_length]['x'].append(conf.beadCoord[m, n, 0])
            beads_by_cycle_length[cycle_length]['y'].append(conf.beadCoord[m, n, 1])
            beads_by_cycle_length[cycle_length]['z'].append(conf.beadCoord[m, n, 2])

    # Plot each cycle length group
    for cycle_length in unique_cycle_lengths:
        color_idx = cycle_length_to_color_idx[cycle_length]
        color = f'hsl({(color_idx * 360 / num_unique_lengths) % 360}, 80, 50)'

        coords = beads_by_cycle_length[cycle_length]
        fig.add_trace(go.Scatter3d(
            x=coords['x'],
            y=coords['y'],
            z=coords['z'],
            mode='markers',
            marker=dict(size=3, color=color),
            name=f'cycle length {cycle_length:.0f}'
        ))

    # Configure layout with physics information
    title_text = f"PIMC Worldlines - Config {conf_idx}<br>"

    fig.update_layout(
        title=dict(text=title_text, x=0.5, font=dict(size=14)),
        scene=dict(
            xaxis_title="X (Å)", yaxis_title="Y (Å)", zaxis_title="Z (Å)",
            aspectmode='data', bgcolor='rgba(240,240,240,0.1)',
            xaxis=dict(showgrid=True, gridcolor='lightgray'),
            yaxis=dict(showgrid=True, gridcolor='lightgray'),
            zaxis=dict(showgrid=True, gridcolor='lightgray')
        ),
        showlegend=True, legend=dict(x=0.02, y=0.98),
        width=900, height=700
    )

    # Add physics annotation
    fig.add_annotation(
        text="Each color = one cycle length",
        xref="paper", yref="paper", x=0.02, y=0.02,
        showarrow=False, font=dict(size=10),
        bgcolor="rgba(255,255,255,0.9)", bordercolor="gray", borderwidth=1
    )

    fig.show()

    # Print physics summary
    print(f"\n=== Configuration {conf_idx} Physics Summary ===")
    print(f"Box: {box[0]} × {box[1]} × {box[2]} Å")
    print(f"total num of cycles: {conf.cycleSizeDist.shape[0]}")
    print(f"unique cycle lengths: {unique_cycle_lengths}")
    print(f"cycles distribution: {conf.cycleSizeDist/conf.numTimeSlices}")

print("✅ Main visualization function ready")

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATION
# ============================================================================

print("🎯 Interactive PIMC Worldline Visualization")

# plot_coordSpaceTrace_3d(0)  # Initial plot for config 0
# Create interactive slider
w = interact(plot_coordSpaceTrace_3d, index=IntSlider(min=0, max=len(confs_closed)-1, step=1, value=0));
display(w)